# Multi-Task AST: Cloud GPU Training (Google Colab)

Trains the Multi-Task Audio Spectrogram Transformer (GTZAN music genres + ESC-50 environmental sounds) with
`configs/cloud_train.yaml`: batch 16, float16 mixed precision, no gradient checkpointing, 20 epochs.

**Before running:** *Runtime > Change runtime type > GPU* (T4 or better). Set the options in cell 0, then run the cells in order.

- Checkpoints go to `checkpoints/cloud_run/`. With `USE_DRIVE = True` that folder lives on Google Drive, so after a
  disconnect you can re-run cells 0-3 and training resumes from `latest_checkpoint.pt`.
- Datasets are not uploaded: GTZAN is downloaded from Kaggle with `kagglehub` (if it asks for credentials, add
  `KAGGLE_USERNAME` / `KAGGLE_KEY` in Colab *Secrets*), and ESC-50 is downloaded from GitHub by `scripts/prepare_manifests.py`.

## 0. Settings and project code

Get the code here either by cloning your git repository (`REPO_URL`) or from a zip made locally with
`python scripts/package_for_cloud.py` and uploaded to Google Drive (`PROJECT_ZIP`).

In [ ]:
# ---- Settings ----
REPO_URL = "https://github.com/SOHAM0007-CODER/Audio-Classifier.git"
PROJECT_ZIP = "/content/drive/MyDrive/multitask_ast/multitask_ast_code.zip"  # from scripts/package_for_cloud.py
USE_DRIVE = True  # keep checkpoints on Google Drive so training can resume after a disconnect
DRIVE_DIR = "/content/drive/MyDrive/multitask_ast"
GTZAN_DIR = ""  # existing GTZAN genres_original folder; leave empty to download from Kaggle
PROJECT_DIR = "/content/multitask_ast"

import os
import subprocess
import zipfile
from pathlib import Path

try:
    from google.colab import drive
except ImportError:  # not running on Colab
    drive = None

if drive is not None and (USE_DRIVE or not REPO_URL):
    drive.mount("/content/drive")

if not Path(PROJECT_DIR, "src", "train.py").exists():
    if REPO_URL:
        subprocess.run(["git", "clone", REPO_URL, PROJECT_DIR], check=True)
    else:
        with zipfile.ZipFile(PROJECT_ZIP) as archive:
            archive.extractall(PROJECT_DIR)
os.chdir(PROJECT_DIR)

if USE_DRIVE and drive is not None:
    drive_checkpoints = Path(DRIVE_DIR, "checkpoints")
    drive_checkpoints.mkdir(parents=True, exist_ok=True)
    if not Path("checkpoints").exists():
        Path("checkpoints").symlink_to(drive_checkpoints, target_is_directory=True)
    print("Checkpoints are saved to", drive_checkpoints)

print("Project directory:", os.getcwd())

## 1. Environment check and installation

`requirements.txt` pins `transformers` 5.x: checkpoint weight names follow its AST module layout, so the same major
version must be used for training here and for inference locally. The unit tests run on CPU in about a minute.

In [ ]:
!nvidia-smi
!pip install -q -r requirements.txt kagglehub
!python -c "import torch, transformers; assert torch.cuda.is_available(), 'No CUDA GPU: Runtime > Change runtime type > GPU'; print('torch', torch.__version__, '| transformers', transformers.__version__, '|', torch.cuda.get_device_name(0), round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1), 'GiB')"
# Pre-flight unit tests: model routing and losses, training loop, metrics and checkpointing.
!python -m pytest -q tests/test_model.py tests/test_train.py

## 2. Data preparation and verification

Downloads GTZAN (unless `GTZAN_DIR` is set), then `scripts/prepare_manifests.py` resamples everything to 16 kHz mono,
slices GTZAN tracks into 5 s chunks with a song-level split, downloads ESC-50, and writes `data/manifests/*.csv`.
The dataset tests then check the manifests, label ranges, feature shapes and that no GTZAN song leaks across splits.

In [ ]:
from pathlib import Path

import pandas as pd

gtzan_dir = GTZAN_DIR
if not gtzan_dir:
    import kagglehub

    dataset_path = Path(kagglehub.dataset_download("andradaolteanu/gtzan-dataset-music-genre-classification"))
    matches = sorted(dataset_path.rglob("genres_original"))
    assert matches, f"genres_original not found under {dataset_path}"
    gtzan_dir = str(matches[0])
print("GTZAN:", gtzan_dir, "|", len(list(Path(gtzan_dir).glob("*/*.wav"))), "tracks")

if not Path("data/manifests/train.csv").exists():
    !python scripts/prepare_manifests.py --gtzan-dir "{gtzan_dir}"

for split in ("train", "val", "test"):
    counts = pd.read_csv(f"data/manifests/{split}.csv")["domain"].value_counts()
    print(f"{split:>5}: music={counts.get(0, 0)}  env={counts.get(1, 0)}")
!python -m pytest -q tests/test_dataset.py

## 3. Training

Logs losses and learning rates every 50 steps and validation metrics after every epoch; `best_model.pt` tracks the
best `combined_metric` and the test split is evaluated with it at the end. Re-running this cell resumes from
`latest_checkpoint.pt` (`--resume auto`). If CUDA runs out of memory, add `--batch_size 8 --grad_accum_steps 2`.

In [ ]:
import os

num_workers = min(4, os.cpu_count() or 1)  # free Colab machines have 2 vCPUs
!python -m src.train --config configs/cloud_train.yaml --resume auto --num_workers {num_workers}

## 4. Test-set evaluation and inference demo

In [ ]:
import json

import pandas as pd

CHECKPOINT = "checkpoints/cloud_run/best_model.pt"

# Standalone test-set evaluation of the best checkpoint.
!python -m src.evaluate --checkpoint {CHECKPOINT} --manifest data/manifests/test.csv --fp16 --output checkpoints/cloud_run/test_metrics.json

# Inference demo on one music and one environmental test clip.
test_df = pd.read_csv("data/manifests/test.csv")
class_names = {}
for domain, mapping_file in ((0, "data/manifests/music_classes.json"), (1, "data/manifests/env_classes.json")):
    with open(mapping_file) as f:
        class_names[domain] = {index: name for name, index in json.load(f).items()}

for domain, domain_flag in ((0, "music"), (1, "env")):
    row = test_df[test_df["domain"] == domain].iloc[0]
    clip = row["filepath"]
    print(f"\n=== {clip} (true label: {class_names[domain][int(row['label'])]}) ===")
    !python -m src.predict --audio "{clip}" --checkpoint {CHECKPOINT} --domain {domain_flag} --top_k 5

## Using the trained model locally

`best_model.pt` (about 330 MB) is in `DRIVE_DIR/checkpoints/cloud_run/` (or download it from `checkpoints/cloud_run/`).
Copy it to `checkpoints/best_model.pt` in the local project and run:

```bash
python -m src.predict --audio clip.wav
```